# PyLabRobot in 3D: motion playback and collision detection

A five-minute tour of two open pull requests against `pylabrobot:main`:

- **[#1468](https://github.com/PyLabRobot/pylabrobot/pull/1468) — motion playback.** Every firmware command a simulated STAR sends is decoded into what its drives do, played in the 3D viewer with measured motion profiles, and held until the page has played it.
- **[#1470](https://github.com/PyLabRobot/pylabrobot/pull/1470) — collision detection.** Channel, 96-head and iSWAP moves are swept against the deck *before* they are sent. A command that would hit something raises `CollisionError`, never reaches the device, and the page draws what it would have hit.

Everything below is a simulated STARlet: no hardware needed. Keep the viewer tab and this notebook side by side.

## The deck

A tip carrier, a source plate and a destination plate. Plain PyLabRobot: nothing here knows about the viewer.

In [ ]:
import logging

from pylabrobot.hamilton.star.device import STARlet
from pylabrobot.resources import set_volume_tracking
from pylabrobot.resources.coordinate import Coordinate
from pylabrobot.resources.corning import cor_96_wellplate_360uL_Fb
from pylabrobot.resources.hamilton import (
  hamilton_96_tiprack_300uL_filter,
  hamilton_plate_carrier_L5_ac,
  hamilton_tip_carrier_L5,
)
from pylabrobot.resources.tip_tracking import set_tip_tracking
from pylabrobot.visualizer3D.facility import Facility

logging.getLogger("pylabrobot").setLevel(logging.WARNING)
set_volume_tracking(True)
set_tip_tracking(True)

facility = Facility(name="lab", size_x=2000, size_y=1200, size_z=1000)
star = STARlet(simulation=True)
facility.assign_child_resource(star, location=Coordinate(0, 0, 0))

tip_carrier = hamilton_tip_carrier_L5(name="tip_carrier")
tip_carrier[0] = tips = hamilton_96_tiprack_300uL_filter(name="tips")
star.deck.assign_child_resource(tip_carrier, track=1)

source_carrier = hamilton_plate_carrier_L5_ac(name="source_carrier")
source_carrier[1] = source = cor_96_wellplate_360uL_Fb(name="source")
star.deck.assign_child_resource(source_carrier, track=8)

destination_carrier = hamilton_plate_carrier_L5_ac(name="destination_carrier")
destination_carrier[1] = destination = cor_96_wellplate_360uL_Fb(name="destination")
star.deck.assign_child_resource(destination_carrier, track=14)

for well in source.get_all_items():
  well.tracker.set_volume(200)

await star.setup()

## Attach the viewer

`open_viewer` starts a `Viewer3D` and wires in the two hooks the PRs add:

```python
attach_viewer_motion(star.driver, viewer)                 # 1468: play each command
attach_viewer_collisions(star.driver, viewer, transport)  # 1470: gate each command
```

`raise_on_collision=True` is what turns the collision checks into gates. Tip: add `?motion=2` to the page address for double speed.

In [ ]:
from viz3d_demo.session import open_viewer

session = await open_viewer(facility, star)
pipettes, transport = star.pipettes, session.transport

## 1 · Pipetting, played as the firmware moves

Each command waits for the page to finish playing it, so the run goes at the pace of the picture: the arm travels in X while the channels spread in Y, drop, and come back up. Volumes and tips are tracked as they go.

In [ ]:
await pipettes.pick_up_tips(tips["A1:H1"])
await pipettes.aspirate(source["A1:H1"], [50] * 8)
await pipettes.dispense(destination["A1:H1"], [50] * 8)

print("destination A1 holds", destination.get_item("A1").tracker.get_used_volume(), "uL")

## 2 · A crash that never happens

The channels are still over the destination plate, carrying tips. Ask them to go down to 150 mm, well below the top of the plate.

The gate sweeps the move against the deck before the firmware command is sent. It refuses, names the command, and the page draws red boxes over what the tips would have hit.

In [ ]:
from pylabrobot.hamilton.star.driver.features.star_collisions import CollisionError

try:
  await pipettes.move_stop_disc_to_z_positions({channel: 150.0 for channel in range(8)})
except CollisionError as refused:
  print("refused:", refused.origin)
  print("would hit:", sorted({c.obstacle.name for c in refused.collisions}))

## 3 · Moving a plate with the iSWAP

`iSWAPTransport` plans the move from where things actually are on the deck, judges the whole plan for collisions, then plays it step by step: reach, grip, carry, set down.

In [ ]:
await pipettes.drop_tips(tips["A1:H1"])
await transport.move_resource(destination, source_carrier[3])

print("destination now sits on", destination.parent.name)

## 4 · A packed deck

Shove the source carrier one track to the left, flush against the tip carrier, the way a crowded deck often ends up. Then ask the iSWAP to pick up the source plate.

The plan is judged before the arm moves: there is no room for the gripper fingers beside the plate. Nothing moves, and the plate stays where it was.

In [ ]:
from pylabrobot.hamilton.star.driver.features.iswap_transport import iSWAPCollisionError

star.deck.unassign_child_resource(source_carrier)
star.deck.assign_child_resource(source_carrier, track=7)

try:
  await transport.move_resource(source, destination_carrier[3])
except iSWAPCollisionError as refused:
  hit = sorted({c.obstacle.name for c in refused.collisions})
  print("plan refused, the fingers would hit:", hit)
print("source is still on", source.parent.name)

## 5 · Fix the deck, run it again

Give the carrier its track back and the exact same call goes through.

In [ ]:
star.deck.unassign_child_resource(source_carrier)
star.deck.assign_child_resource(source_carrier, track=8)

await transport.move_resource(source, destination_carrier[3])
print("source now sits on", source.parent.name)

## What's under the hood

| | |
|---|---|
| `hamilton/star/motion.py` | decodes each firmware command into drive targets and timings (`MOTION_PROFILES.md`) |
| `resources/collision.py` | machine-agnostic engine: convex pieces, swept hulls, GJK distance, a broadphase |
| `star_collisions.py` / `iswap_collisions.py` | what moves with each STAR command, and what it is allowed to touch |
| `iswap_transport.py` | iSWAP plate and lid moves planned from the resource tree |

With no viewer attached, a simulated run sends the same commands it does on `main`. The gate only stands between the driver and the device when a viewer opts in with `raise_on_collision=True`.